# EDA

In [1]:
import requests
from bs4 import BeautifulSoup as bs

import pandas as pd
import numpy as np

import seaborn as sns

import plotly
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go


from io import StringIO

In [2]:
# получаем список системно значимых банков

link = 'https://www.cbr.ru/banking_sector/credit/systembanks.html/'

html = requests.get(link)
if html.status_code == 200:
    soup = bs(html.text, 'lxml')


In [3]:
table = soup.find('table', class_ = 'data')
table = str(table)

table = StringIO(table)
df_sszn = pd.read_html(table)[0]

In [4]:
df_sszn

,№ п/п,Наименование кредитной организации,Рег. №
0,1,АО ЮниКредит Банк,1
1,2,Банк ГПБ (АО),354
2,3,ПАО «Совкомбанк»,963
3,4,Банк ВТБ (ПАО),1000
4,5,АО «АЛЬФА-БАНК»,1326
5,6,ПАО Сбербанк,1481
6,7,ПАО «Московский Кредитный Банк»,1978
7,8,ПАО Банк «ФК Открытие»,2209
8,9,ПАО РОСБАНК,2272
9,10,АО «ТБанк»,2673


In [5]:
regn_list = df_sszn['Рег. №'].to_list()
regn_list = [str(x) for x in regn_list]
regn_list

['1',
 '354',
 '963',
 '1000',
 '1326',
 '1481',
 '1978',
 '2209',
 '2272',
 '2673',
 '3251',
 '3292',
 '3349']

In [6]:
df = pd.read_csv('output/2025-03-12.csv', index_col=0)
df.head()

,min_days_value,max_days_value,min_value,max_value,rate_value,bank_name,deposit_name,tool_type,regn,deposit_link,Пополнение счета,Частичное снятие,Пролонгация,date_parse
0,91,180,30000.0,1500000.0,22.6,Банк ДОМ.РФ,Мой ДОМ,Вклад,2312,https://bankiros.ru/bank/roscap/deposits/moj-d...,Нет,Нет,Есть,2025-03-12
1,91,180,1500000.0,15000000.0,21.9,Банк ДОМ.РФ,Мой ДОМ,Вклад,2312,https://bankiros.ru/bank/roscap/deposits/moj-d...,Нет,Нет,Есть,2025-03-12
2,181,273,30000.0,1500000.0,20.9,Банк ДОМ.РФ,Мой ДОМ,Вклад,2312,https://bankiros.ru/bank/roscap/deposits/moj-d...,Нет,Нет,Есть,2025-03-12
3,181,273,1500000.0,15000000.0,20.6,Банк ДОМ.РФ,Мой ДОМ,Вклад,2312,https://bankiros.ru/bank/roscap/deposits/moj-d...,Нет,Нет,Есть,2025-03-12
4,274,366,30000.0,1500000.0,17.1,Банк ДОМ.РФ,Мой ДОМ,Вклад,2312,https://bankiros.ru/bank/roscap/deposits/moj-d...,Нет,Нет,Есть,2025-03-12


In [7]:
df_loc = df[
    (
        (df['min_days_value'].between(340, 370)) | (df['max_days_value'].between(340, 370))
    ) &
    (
        # (df['min_value'] < 900000) | (df['max_value'] > 900000)
        (df['min_value'] <= 1000000) & (df['max_value'] >= 1000000)
    ) &
    (
        df['Пополнение счета'] == 'Нет'
    ) &
    (
        df['Частичное снятие'] == 'Нет'
    ) &
    (
        df['Пролонгация'] == 'Нет'
    )
    ]

df_antiloc = df[~df.isin(df_loc)].dropna()


In [21]:
df_loc_max = df_loc.loc[df_loc.groupby(["bank_name"])["rate_value"].idxmax()]
df_antiloc_max = df_antiloc.loc[df_antiloc.groupby(["bank_name"])["rate_value"].idxmax()]
# df_loc_max

In [18]:
df_max = df.loc[df["bank_name"].isin(df_loc_max['bank_name'])]
df_max = df_max.loc[df_max.groupby(["bank_name"])["rate_value"].idxmax()]
df_max

,min_days_value,max_days_value,min_value,max_value,rate_value,bank_name,deposit_name,tool_type,regn,deposit_link,Пополнение счета,Частичное снятие,Пролонгация,date_parse
4338,7,365,100000.0,999999999.0,28.00,Lender Invest,Защита капитала,Вклад,нет регномера,https://bankiros.ru/bank/lender-invest/deposit...,Нет,Нет,Нет,2025-03-12
514,271,360,10000.0,999999999.0,26.00,Абсолют Банк,Турбо,Вклад,2306,https://bankiros.ru/bank/absolutbank/deposits/...,Нет,Нет,Есть,2025-03-12
4228,367,9999,10000.0,999999999.0,17.50,Авангард,Базовый-Интернет (проценты в конце срока),Вклад,2879,https://bankiros.ru/bank/avangard/deposits/baz...,Нет,Нет,Нет,2025-03-12
4239,184,9999,10000.0,999999999.0,16.25,Аверс,Гармония,Вклад,415,https://bankiros.ru/bank/aversbank/deposits/ga...,Нет,Нет,Нет,2025-03-12
2686,180,9999,100000.0,999999999.0,20.50,Агора,с ежемесячными выплатами,Вклад,3231,https://bankiros.ru/bank/agora/deposits/vklad-...,Нет,Нет,Нет,2025-03-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2818,91,180,1000000.0,999999999.0,20.00,ЧБРР,Клевер2+ VIP2,Вклад,3527,https://bankiros.ru/bank/chbrr/deposits/klever...,Нет,Нет,Нет,2025-03-12
2135,181,270,10000.0,999999999.0,21.00,Челябинвестбанк,Золотой максимум,Вклад,493,https://bankiros.ru/bank/chelinvest/deposits/z...,Нет,Нет,Нет,2025-03-12
3659,365,9999,0.0,999999999.0,3.00,Эко-Инвест,Срочный,Вклад,3116,https://bankiros.ru/bank/eco-invest/deposits/s...,Нет,Нет,Нет,2025-03-12
1926,181,272,50000.0,999999999.0,21.50,Экономбанк,Доходный,Вклад,1319,https://bankiros.ru/bank/econombank/deposits/d...,Нет,Нет,Нет,2025-03-12


In [19]:
df_loc_max_merge = df_loc_max.merge(df_max, on = 'bank_name')
df_loc_max_merge

,min_days_value_x,max_days_value_x,min_value_x,max_value_x,rate_value_x,bank_name,deposit_name_x,tool_type_x,regn_x,deposit_link_x,...,max_value_y,rate_value_y,deposit_name_y,tool_type_y,regn_y,deposit_link_y,Пополнение счета_y,Частичное снятие_y,Пролонгация_y,date_parse_y
0,7,365,100000.0,999999999.0,28.0,Lender Invest,Защита капитала,Вклад,нет регномера,https://bankiros.ru/bank/lender-invest/deposit...,...,999999999.0,28.00,Защита капитала,Вклад,нет регномера,https://bankiros.ru/bank/lender-invest/deposit...,Нет,Нет,Нет,2025-03-12
1,367,540,10000.0,1399990.0,17.0,Абсолют Банк,Авансовый,Вклад,2306,https://bankiros.ru/bank/absolutbank/deposits/...,...,999999999.0,26.00,Турбо,Вклад,2306,https://bankiros.ru/bank/absolutbank/deposits/...,Нет,Нет,Есть,2025-03-12
2,367,9999,10000.0,999999999.0,17.5,Авангард,Базовый-Интернет (проценты в конце срока),Вклад,2879,https://bankiros.ru/bank/avangard/deposits/baz...,...,999999999.0,17.50,Базовый-Интернет (проценты в конце срока),Вклад,2879,https://bankiros.ru/bank/avangard/deposits/baz...,Нет,Нет,Нет,2025-03-12
3,367,9999,10000.0,999999999.0,15.0,Аверс,Традиция,Вклад,415,https://bankiros.ru/bank/aversbank/deposits/tr...,...,999999999.0,16.25,Гармония,Вклад,415,https://bankiros.ru/bank/aversbank/deposits/ga...,Нет,Нет,Нет,2025-03-12
4,365,9999,500000.0,5000000.0,16.0,Агора,на год,Вклад,3231,https://bankiros.ru/bank/agora/deposits/vklad,...,999999999.0,20.50,с ежемесячными выплатами,Вклад,3231,https://bankiros.ru/bank/agora/deposits/vklad-...,Нет,Нет,Нет,2025-03-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,181,366,1000000.0,999999999.0,20.0,ЧБРР,Клевер2+ VIP2,Вклад,3527,https://bankiros.ru/bank/chbrr/deposits/klever...,...,999999999.0,20.00,Клевер2+ VIP2,Вклад,3527,https://bankiros.ru/bank/chbrr/deposits/klever...,Нет,Нет,Нет,2025-03-12
66,271,360,10000.0,999999999.0,19.0,Челябинвестбанк,Золотой максимум,Вклад,493,https://bankiros.ru/bank/chelinvest/deposits/z...,...,999999999.0,21.00,Золотой максимум,Вклад,493,https://bankiros.ru/bank/chelinvest/deposits/z...,Нет,Нет,Нет,2025-03-12
67,365,9999,0.0,999999999.0,3.0,Эко-Инвест,Срочный,Вклад,3116,https://bankiros.ru/bank/eco-invest/deposits/s...,...,999999999.0,3.00,Срочный,Вклад,3116,https://bankiros.ru/bank/eco-invest/deposits/s...,Нет,Нет,Нет,2025-03-12
68,273,364,50000.0,999999999.0,19.7,Экономбанк,Доходный,Вклад,1319,https://bankiros.ru/bank/econombank/deposits/d...,...,999999999.0,21.50,Доходный,Вклад,1319,https://bankiros.ru/bank/econombank/deposits/d...,Нет,Нет,Нет,2025-03-12


In [26]:
df_loc_max_merge.to_excel('output/df_loc_max_merge.xlsx')
df_antiloc_max.to_excel('output/df_antiloc_max.xlsx')

In [11]:
df_loc_max_merge

,min_days_value,max_days_value,min_value,max_value,rate_value,bank_name,deposit_name,tool_type,regn,deposit_link,Пополнение счета,Частичное снятие,Пролонгация,date_parse
0,7,365,100000.0,999999999.0,28.0,Lender Invest,Защита капитала,Вклад,нет регномера,https://bankiros.ru/bank/lender-invest/deposit...,Нет,Нет,Нет,2025-03-12
1,367,540,10000.0,1399990.0,17.0,Абсолют Банк,Авансовый,Вклад,2306,https://bankiros.ru/bank/absolutbank/deposits/...,Нет,Нет,Нет,2025-03-12
2,367,9999,10000.0,999999999.0,17.5,Авангард,Базовый-Интернет (проценты в конце срока),Вклад,2879,https://bankiros.ru/bank/avangard/deposits/baz...,Нет,Нет,Нет,2025-03-12
3,367,9999,10000.0,999999999.0,15.0,Аверс,Традиция,Вклад,415,https://bankiros.ru/bank/aversbank/deposits/tr...,Нет,Нет,Нет,2025-03-12
4,365,9999,500000.0,5000000.0,16.0,Агора,на год,Вклад,3231,https://bankiros.ru/bank/agora/deposits/vklad,Нет,Нет,Нет,2025-03-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,181,366,1000000.0,999999999.0,20.0,ЧБРР,Клевер2+ VIP2,Вклад,3527,https://bankiros.ru/bank/chbrr/deposits/klever...,Нет,Нет,Нет,2025-03-12
66,271,360,10000.0,999999999.0,19.0,Челябинвестбанк,Золотой максимум,Вклад,493,https://bankiros.ru/bank/chelinvest/deposits/z...,Нет,Нет,Нет,2025-03-12
67,365,9999,0.0,999999999.0,3.0,Эко-Инвест,Срочный,Вклад,3116,https://bankiros.ru/bank/eco-invest/deposits/s...,Нет,Нет,Нет,2025-03-12
68,273,364,50000.0,999999999.0,19.7,Экономбанк,Доходный,Вклад,1319,https://bankiros.ru/bank/econombank/deposits/d...,Нет,Нет,Нет,2025-03-12


In [ ]:


df_loc.to_csv('output/loc_filter_v3.csv')
df_antiloc.to_csv('output/antiloc_filter_v3.csv')


In [ ]:
df_loc.to_csv('output/filter_340_370_1000000.csv')

In [ ]:
# gantt_df = df[['deposit_link','regn', 'bank_name', 'deposit_name', 'min_value', 'max_value' ,'min_days_value', 'max_days_value']]
df_loc['sszn_check'] = df_loc['regn'].apply(lambda x: 1 if x in regn_list else 0)
gantt_df = df_loc[df_loc['sszn_check']==1]

gantt_df['task_name'] = gantt_df['bank_name'] + ' // ' + gantt_df['min_value'].astype(str) + ' // '+ gantt_df['max_value'].astype(str)

gantt_df_loc = gantt_df[['task_name', 'min_days_value', 'max_days_value', 'regn']]

# gantt_df_loc

gantt_df_loc['max_days_value_custom'] = gantt_df_loc['max_days_value'].apply(lambda x: 1800 if x == 9999 else x)
# gantt_df_loc['max_days_value_custom'] = np.where(gantt_df_loc['max_days_value'] == 9999, 1800, gantt_df_loc['max_days_value'])

In [ ]:
test_df = gantt_df_loc[gantt_df_loc['task_name']=='Т-Банк // 50000.0 // 999999999.0']
test_df = test_df.rename(columns={'task_name':'Task', 'min_days_value':'Start', 'max_days_value_custom':'Finish'})

fig = ff.create_gantt(
    test_df, 
    # x_start="min_days_value",
    # x_end="max_days_value", 
    # y="task_name",
    group_tasks=True,
    index_col = 'regn',
    bar_width = 0.4, 
    show_colorbar=False)

fig.update_layout(
    xaxis_type='linear', 
    autosize=False, 
    width=1200, 
    height=300)

fig.update_traces(opacity=0.5, selector={'fill':'toself'})

fig.show()

In [ ]:
gantt_df_gantt = gantt_df_loc.rename(columns={'task_name':'Task', 'min_days_value':'Start', 'max_days_value_custom':'Finish'})

In [ ]:
fig = ff.create_gantt(
    gantt_df_gantt, 
    # x_start="min_days_value",
    # x_end="max_days_value", 
    # y="task_name",
    index_col = 'regn',
    bar_width = 0.4, 
    show_colorbar=False,
    group_tasks=True,)

fig.update_layout(
    xaxis_type='linear', 
    autosize=False, 
    width=1200, 
    height=800,
    title="Наглядный Гантт с областями"
    )

fig.update_traces(opacity=0.5, selector={'fill':'toself'})

# print(fig['layout'])

# fig.add_trace(
#     go.Scatter(
#         x = [350, 400],
#         y = [-1, len(gantt_df_gantt.index) + 1],
#         mode = "lines",
#         line = go.scatter.Line(color = "gray", width = 1),
#         showlegend = False
#     )
# )

fig.add_vrect(
    x0=350, 
    x1=400, 
    line_width=0, fillcolor="red", opacity=0.2)

fig.show()

In [ ]:
fig.write_html("output/gantt_v_02.html")

In [ ]:
# for i in gantt_df['task_name'].unique():
#     temp_df = gantt_df[gantt_df['task_name']==i]
#     print(temp_df)
    
#     break

In [ ]:
# df['bank_name'].unique()